# Recalllens evaluation
This notebook runs the production-equivalent OpenCV 5 matcher, summarizes accuracy and latency, and examines GPS/time as evaluation covariates.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
subprocess.run([sys.executable, str(ROOT / 'src/recall_lens_eval.py')], check=True)

In [ ]:
summary = json.loads((ROOT / 'output/summary.json').read_text())
summary

In [ ]:
predictions = pd.read_csv(ROOT / 'output/predictions.csv')
comparisons = pd.read_csv(ROOT / 'output/comparisons.csv')
metadata = pd.read_csv(ROOT / 'output/image_metadata.csv')
predictions.groupby(['split', 'condition'], dropna=False)['correct'].agg(['count', 'mean'])

In [ ]:
positive_scores = predictions.loc[predictions.expected_match, 'top_score']
negative_scores = predictions.loc[~predictions.expected_match, 'top_score']
plt.hist([positive_scores, negative_scores], bins=20, label=['positive', 'negative'], alpha=.75)
plt.axvline(summary['threshold'], color='black', linestyle='--', label='production threshold')
plt.xlabel('Top similarity score'); plt.ylabel('Queries'); plt.legend(); plt.title('Score separation');

In [ ]:
thresholds = [value / 100 for value in range(0, 101)]
curve = []
for threshold in thresholds:
    positives = predictions[predictions.expected_match]
    negatives = predictions[~predictions.expected_match]
    tur = ((positives.top_score >= threshold) & (positives.predicted_scene == positives.expected_scene)).mean()
    fur = (negatives.top_score >= threshold).mean()
    curve.append((threshold, tur, fur))
curve = pd.DataFrame(curve, columns=['threshold', 'true_unlock_rate', 'false_unlock_rate'])
curve.plot(x='threshold', y=['true_unlock_rate', 'false_unlock_rate'], ylim=(0, 1), grid=True);

In [ ]:
display(metadata[['path', 'role', 'scene', 'captured_at', 'latitude', 'longitude', 'camera_model']])
display(predictions.loc[~predictions.correct].sort_values('top_score', ascending=False))